In [1]:
HOPWORKS_PROJECT_NAME = 'sucram_taxi_predictor'

In [16]:
import os
from dotenv import load_dotenv
from src.paths import PARENT_DIR

load_dotenv(dotenv_path=os.path.join(PARENT_DIR, '.env'))

HOPWORKS_API_KEY = os.getenv('HOPWORKS_API_KEY')

## Fetch raw data

In [3]:
from datetime import datetime
import pandas as pd
from src.data import load_raw_data

from_year = 2022
to_year = datetime.now().year
print(f'Fetching data from {from_year} to {to_year}')

rides = pd.DataFrame()
for year in range(from_year, to_year + 1):
    rides_of_year = load_raw_data(year)
    rides = pd.concat([rides, rides_of_year])


Fetching data from 2022 to 2024
File 2022-01 was already in local storage
File 2022-02 was already in local storage
File 2022-03 was already in local storage
File 2022-04 was already in local storage
File 2022-05 was already in local storage
File 2022-06 was already in local storage
File 2022-07 was already in local storage
File 2022-08 was already in local storage
File 2022-09 was already in local storage
File 2022-10 was already in local storage
File 2022-11 was already in local storage
File 2022-12 was already in local storage
File 2023-01 was already in local storage
File 2023-02 was already in local storage
File 2023-03 was already in local storage
File 2023-04 was already in local storage
File 2023-05 was already in local storage
File 2023-06 was already in local storage
File 2023-07 was already in local storage
File 2023-08 was already in local storage
File 2023-09 was already in local storage
File 2023-10 was already in local storage
File 2023-11 was already in local storage
Fi

In [4]:
print(f'Fetched {rides.shape[0]} rides')

Fetched 80928741 rides


In [5]:
from src.data import transform_raw_data_into_ts_data

ts_data = transform_raw_data_into_ts_data(rides)

100%|██████████| 263/263 [00:04<00:00, 63.88it/s]


In [11]:
import hopsworks

In [18]:
project = hopsworks.login(
    project=HOPWORKS_PROJECT_NAME,
    api_key_value=HOPWORKS_API_KEY
)

Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/630933


In [19]:
feature_store = project.get_feature_store()

Connected. Call `.close()` to terminate connection gracefully.


In [20]:
FEATURE_GROUP_NAME = 'time_series_hourly_feature_group'
FEATURE_GROUP_VERSION = 1

In [21]:
feature_group = feature_store.get_or_create_feature_group(
    name=FEATURE_GROUP_NAME,
    version=FEATURE_GROUP_VERSION,
    description='Time series hourly feature group',
    primary_key=['pickup_location_id', 'pickup_hour'],
    event_time='pickup_hour',
)

In [22]:
feature_group.insert(ts_data, write_options={"wait_for_job": False})

Feature Group created successfully, explore it at 
https://c.app.hopsworks.ai:443/p/630933/fs/626756/fg/715092


Uploading Dataframe: 0.00% |          | Rows 0/4803432 | Elapsed Time: 00:00 | Remaining Time: ?

Launching job: time_series_hourly_feature_group_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai/p/630933/jobs/named/time_series_hourly_feature_group_1_offline_fg_materialization/executions


(<hsfs.core.job.Job at 0x11f1db990>, None)